# TP 1 Architectures Logicielles & qualités

Ce notebook présente un modèle multi-agent basé sur le modèle SEIR (Susceptible-Exposed-Infected-Recovered) pour la simulation de la propagation des maladies. L'objectif est de comprendre le processus d'infection entre individus dans une situation d'épidémie. Nous décrirons la conception, et l'analyse des résultats issus de 100 réplications de 730 jours, l'implémentation ne sera pas détaillée ici mais les fichiers C++ seront disponibles sur le Git associé. Le code sera documenté par la syntaxe de Doxygen. L'objectif est d'approfondir la compréhension des phases de la propagation des maladies.

Commençons par compiler notre simulation :

/!\ Pensez à installer gcc (g++) et cmake si ce n'est pas deja fait !

## Compilation

In [ ]:
#  ___  _   _  ___  _     ___  
# | _ )| | | ||_ _|| |   |   \ 
# | _ \| |_| | | | | |__ | |) |
# |___/ \___/ |___||____||___/ 
                        

!cmake .
!cmake --build .

On lance maintenant notre programme. L'exécution peut prendre quelques instant. En cas de problème, il est possible de lancer l'exécutable TP1 à la main.

La reproductibilité du code est assurée par notre seed fixée de Mersenne Twister.
    On utilise l'implémentation standard C++ de ce générateur de nombre pseudo-aléatoires.

## Exécution

In [ ]:
#  ___  _   _  _  _ 
# | _ \| | | || \| |
# |   /| |_| || .  |
# |_|_\ \___/ |_|\_|


from subprocess import Popen

p = Popen("./TP1", stdout=-1)

print("[", end="")

for i in p.stdout:
    print("|", end='')

print("]")

## Chargement

In [ ]:
#  _      ___   ___  ___  
# | |    / _ \ /   \|   \ 
# | |__ | (_) || - || |) |
# |____| \___/ |_|_||___/ 

import matplotlib.pyplot as plt
import os, csv, numpy as np

from os.path import isdir


dir = "out/"
# find the out/ dir
if isdir("out"): pass
elif isdir("cmake-build-debug/out"): dir = "cmake-build-debug/out/" # CLion build folder
elif isdir("../out"): dir = "../out"
 
    
data = []
    
print("Loading: [", end="")

# load data
for i in os.listdir(dir):
    print("|", end="")
    
    # load file n°i
    with open(dir + i, "r") as file:
        data.append(cd := [])
        reader = csv.reader(file, delimiter=";", quoting=csv.QUOTE_NONNUMERIC)
        cd.extend(l for l in reader)
        
print("]")

data = np.array(data) # cast to numpy array

## Analyse


Ce premier graphique représente les courbes de nos 100 simulations :

In [ ]:

plt.figure(figsize=(16, 10)) # set plot size

# plot all curves
for i in range(100):
    x = range(730)
    a = 1 - i / 200         # define sliding color
    alpha = 0.2 if i else 1 # alpha to 1 when i = 0 -> color used by the legend
    
    colors = ((a, 0, 0, alpha), (a/2, a, 0, alpha), (a, 0, a, alpha), (0, a, a, alpha))

    for u in range(4): # plot each status for the simu n°i
        plt.plot(x, data[i,:, u], color=colors[u])
        

plt.title("Evolution of an infection in a humans population (100 superposed simulations)")
plt.xlabel("days")
plt.ylabel("humans")
plt.legend(['Susceptible', 'Exposed', 'Infected', 'Recovered'], prop={'size': 20})

On observe que la plupart des courbes ont des tendances similaires. En effet, elles comptent quelques rebond épidémiques avant de se lisser petit à petit. D'ailleurs, elles semblent converger vers une situation stable. Cependant, certaines courbes forment une sorte de croix. On peut déduire que dans ces quelques cas, le virus s'éteint.

Calculons la fréquence de cette situation :

In [ ]:
counter = sum(data[i,729,2] == 0 for i in range(100))
print(f"Le Virus s'éteint dans {counter}% des cas ")

Pour simplifier la lecture, on effectue un moyennage jour par jour des différentes courbes. Voici le résultat :

In [ ]:

plt.figure(figsize=(16, 10)) # set plot size

a = 1
colors = ((a, 0, 0), (a/2, a, 0), (a, 0, a), (0, a, a))

# plot the average curve of each status
for u in range(4):
    plt.plot(x, data[:,:, u].mean(axis=0), color=colors[u], linestyle='dashed')
        

plt.title("Average evolution of an infection in a humans population")
plt.xlabel("days")
plt.ylabel("humans")
plt.legend(['Susceptible', 'Exposed', 'Infected', 'Recovered'], prop={'size': 20})

Voici le diagramme de classe UML représentant la structure de notre simulation.
Certains compromis sont susceptibles d'être fait mais nous avons choisi de conserver les plus proches de l'implémentation afin que la lecture soit plus claire.
C'est par exemple le cas de la composition de Simulation, en effet, le tableau humans est un tableau de pointeurs comme on le voit dans l'attribut. Il contient 20 000 humains comme on le voit sur la flèche de composition.

![Diagramme de CLasses](UML.png)

In [ ]:
import matplotlib.pyplot as plt

for i in range(100):
    plt.figure(figsize=(16, 10))  # Create a new figure for each iteration

    x = range(730)
    a = 1 - i / 200         # Define sliding color
    alpha = 0.2 if i else 1 # Alpha to 1 when i = 0 -> color used by the legend
    
    colors = ((a, 0, 0, alpha), (a/2, a, 0, alpha), (a, 0, a, alpha), (0, a, a, alpha))

    for u in range(4):  # Plot each status for the simu n°i
        plt.plot(x, data[i, :, u], color=colors[u])

    plt.title(f"Evolution of an infection in a humans population (Simulation #{i+1})")
    plt.xlabel("Days")
    plt.ylabel("Humans")
    plt.legend(['Susceptible', 'Exposed', 'Infected', 'Recovered'], prop={'size': 20})

    plt.show()  # Show the current figure

    # Optionally, save each figure to a file
    # plt.savefig(f"simulation_{i+1}.png")
